In [ ]:
import kagglehub
import pandas as pd
import duckdb
from pathlib import Path
import matplotlib.pyplot as plt

c:\Users\ASUS\Documents\DS Projects\Kaggle Landing Page AB Testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#get data from kaggle
path = kagglehub.dataset_download("zhangluyuan/ab-testing")
df = pd.read_csv(Path(path) / "ab_data.csv")

df.head()


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [3]:
#find out if type conversions are necessary
#look at initial memory usage
df.info(memory_usage = "deep")

<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 57.6 MB


In [4]:
#find out if downcasting integers is viable
df.describe() 

,user_id,converted
count,294478.000000,294478.000000
mean,787974.124733,0.119659
std,91210.823776,0.324563
min,630000.000000,0.000000
25%,709032.250000,0.000000
50%,787933.500000,0.000000
75%,866911.750000,0.000000
max,945999.000000,1.000000


DATA CLEANING:

#Casting: 

- 'user_id' column can be downcast to an int32
- 'timestamp' column needs to be converted to an actual timestamp
- 'group' and 'landing_page' columns can be converted to categories
- 'converted' column could be converted to a boolean

#Further info obtained from data wrangler:

- no null or erroneous values in any column
- many repeated user_id's (~1%) - find out if it's okay to keep them

In [5]:
df['user_id'] = df['user_id'].astype('int32')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['group'] = df['group'].astype('category')
df['landing_page'] = df['landing_page'].astype('category')
df['converted'] = df['converted'].astype('bool')

df.info(memory_usage='deep') #memory reduced from 57.6MB to 4.2MB

<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   user_id       294478 non-null  int32         
 1   timestamp     294478 non-null  datetime64[us]
 2   group         294478 non-null  category      
 3   landing_page  294478 non-null  category      
 4   converted     294478 non-null  bool          
dtypes: bool(1), category(2), datetime64[us](1), int32(1)
memory usage: 4.2 MB


In [6]:
dupl = duckdb.sql(
"""
--Remove rows where the the landing page does not correspond to the correct group
WITH valid_group AS(
SELECT * FROM df WHERE ("group" = 'treatment' AND landing_page = 'new_page') OR ("group" = 'control' AND landing_page = 'old_page')
)

--Any remaining duplicate rows will have the same group and landing page, so just keep the one with the earlier timestamp
SELECT * EXCLUDE(rn) FROM(SELECT *, ROW_NUMBER() OVER(partition by user_id ORDER BY timestamp) AS rn FROM valid_group) WHERE rn = 1
"""
).df()

dupl

,user_id,timestamp,group,landing_page,converted
0,835953,2017-01-05 14:33:35.263330,control,old_page,False
1,889632,2017-01-06 01:03:12.034331,treatment,new_page,False
2,935568,2017-01-09 14:04:33.404948,treatment,new_page,False
3,858745,2017-01-04 00:37:56.038668,treatment,new_page,False
4,860522,2017-01-09 20:45:19.317513,control,old_page,True
...,...,...,...,...,...
290579,710727,2017-01-14 09:55:14.392767,treatment,new_page,False
290580,677860,2017-01-13 15:20:20.921697,treatment,new_page,False
290581,921008,2017-01-10 19:11:03.749130,treatment,new_page,False
290582,747348,2017-01-23 11:56:08.098582,treatment,new_page,False


In [18]:
control_conversion_rate = len(df[(df["group"] == "control") & df["converted"] == True]) / len(df[(df["group"] == "control")])
treatment_conversion_rate = len(df[(df["group"] == "treatment") & df["converted"] == True]) / len(df[(df["group"] == "treatment")])

print(control_conversion_rate)
print(treatment_conversion_rate)

0.12039917935897611
0.11891957956489856
